In [1]:
from transformers import AutoTokenizer, AutoConfig, EncoderDecoderModel
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show
import torch
from torch import nn
import torch.nn.functional as F
from math import sqrt

c:\Repositories\proai\course9-genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
MODEL = "patrickvonplaten/bert2bert_cnn_daily_mail"
model = EncoderDecoderModel.from_pretrained(MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [10]:
text = "io sono gabriele e mi piace il sushi"

In [9]:
config = AutoConfig.from_pretrained(MODEL)

In [ ]:
token_emb = nn.Embedding(config.vocab_size, config.decoder.hidden_size)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
embeddings = token_emb(inputs.input_ids)

In [17]:
seq_len = inputs.input_ids.size(-1)
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)

In [21]:
query = key = value = embeddings

dim_k = key.size(-1)
scores = torch.bmm(query, key.transpose(1,2)) / sqrt(dim_k)

In [24]:
scores.masked_fill(mask == 0, -float("inf"))

tensor([[[ 2.8812e+01,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf],
         [-1.9464e-01,  2.6475e+01,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf],
         [-1.0462e+00, -1.7072e-01,  2.8763e+01,        -inf,        -inf,
                 -inf,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf],
         [-1.9674e-01, -1.4858e+00, -5.7087e-01,  2.9242e+01,        -inf,
                 -inf,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf],
         [-6.4068e-02,  1.6539e-01,  1.6899e-02,  1.8579e+00,  2.7871e+01,
                 -inf,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf],
         [ 9.6615e-01, -2.4453e-01,  3.5272e-01,  6.8880e-01,  1.

In [ ]:
weights = F.softmax(scores, dim=-1)
attn = torch.bmm(weights, value)
attn.shape

In [ ]:
class AttentionHead(nn.Module):

    def scaled_dot_product_attn(self, q, k, v, mask=None):
        dim_k = q.size(-1)
        scores = torch.bmm(q, k.transpose(1,2)) / sqrt(dim_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -float("inf"))
        weights = F.softmax(scores, dim=-1)
        attention = torch.bmm(weights, v)

        return attention

    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden, mask=None):
        attn_outputs = self.scaled_dot_product_attn(
            self.q(hidden), self.k(hidden), self.v(hidden), mask = mask
        )
        return attn_outputs

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config, mask=None):
        super().__init__()
        embed_dim = config.decoder.hidden_size #dimensione per gli hidden sizes
        num_heads = config.decoder.num_attention_heads
        head_dim = embed_dim // num_heads

        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )

        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden):
        x = torch.cat([h(hidden) for h in self.heads], dim=-1)
        x = self.output_linear(x)
        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.decoder.hidden_size)
        self.layer_norm_2 = nn.LayerNorm(config.decoder.hidden_size)

        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)